In [ ]:
from google.colab import files
files.upload()   # pick gk04_queries.csv


In [ ]:
# ================================================================
# GK-04 CONTENT MODERATION  ->  TENSORFLOW REPLICA  (clean report)
# Input : reach, account_age_days, months_active, report_ratio
# Output: score (0-1) and decision (APPROVE / DECLINE)
# Needs : /content/gk04_queries.csv with columns
#         reach, account_age_days, months_active, report_ratio, score, decision
# Saves : gk04_tf_model.keras + gk04_preprocessor.joblib + gk04_config.json
#         (these three files together = the trained model)
# ================================================================
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import json, random, warnings
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, average_precision_score, precision_recall_curve,
    confusion_matrix, mean_absolute_error, mean_squared_error, r2_score,
)

warnings.filterwarnings("ignore")

# ---------------- SETTINGS ----------------
SEED = 42
FILE = "/content/gk04_queries.csv"
OUT_DIR = "/content"
if not os.path.isdir(OUT_DIR):            # running locally
    OUT_DIR, FILE = ".", "gk04_queries.csv"

SCHEMA = {
    "reach":            (0.0, 100.0),
    "account_age_days": (18.0, 75.0),
    "months_active":    (0.0, 40.0),
    "report_ratio":     (0.0, 1.0),
}
numeric_features = list(SCHEMA)
THRESHOLD = 0.527        # portal: 0.5099 = DECLINE, 0.5436 = APPROVE

C_OK, C_BAD, C_MAIN, C_ALT = "#2a9d8f", "#e76f51", "#264653", "#e9c46a"

np.random.seed(SEED); random.seed(SEED); tf.random.set_seed(SEED)
plt.rcParams.update({
    "figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titleweight": "bold", "axes.titlesize": 13,
})

def banner(t):
    print("\n" + "=" * 70 + f"\n{t}\n" + "=" * 70)

def fmt(v):
    return "n/a" if v is None or (isinstance(v, float) and np.isnan(v)) else f"{v:.4f}"

def finish(fig, name):
    fig.tight_layout()
    path = os.path.join(OUT_DIR, name)
    fig.savefig(path, dpi=140)
    plt.show()
    print(f"   saved -> {path}")


# ================================================================
# 1. LOAD + CLEAN
# ================================================================
banner("1. DATA")
raw = pd.read_csv(FILE)
need = numeric_features + ["score", "decision"]
miss = [c for c in need if c not in raw.columns]
if miss:
    raise ValueError(f"CSV is missing required columns: {miss}")

df = raw[need].drop_duplicates().reset_index(drop=True)

if df["decision"].dtype == "object":
    df["decision"] = (df["decision"].astype(str).str.strip().str.upper()
                      .map({"APPROVE": 0, "DECLINE": 1}))
df["decision"] = pd.to_numeric(df["decision"], errors="coerce")
df["score"] = pd.to_numeric(df["score"], errors="coerce")
if df[["decision", "score"]].isna().any().any():
    raise ValueError("'score' / 'decision' contain invalid or missing values.")
df["decision"] = df["decision"].astype(int)

n_conf = int((df.groupby(numeric_features)["score"].nunique() > 1).sum())
info = pd.Series({
    "Rows in file": len(raw),
    "Unique rows used": len(df),
    "APPROVE rows": int((df.decision == 0).sum()),
    "DECLINE rows": int((df.decision == 1).sum()),
    "Ignored columns": ", ".join(c for c in raw.columns if c not in need) or "none",
    "Conflicting input combos": n_conf,
})
print(info.to_string())
if n_conf:
    print("\n  ! Same inputs gave different scores (hidden fields were varying)."
          "\n    Expect a noise floor on the error.")

print("\nFeature summary:")
print(df[numeric_features + ["score"]].describe().T[["min", "mean", "max"]].round(3).to_string())

X = df[numeric_features]
y_score = df["score"].values
y_dec = df["decision"].values


# ================================================================
# 2. EXPLORATORY GRAPHS
# ================================================================
banner("2. DATA GRAPHS")

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(df.loc[df.decision == 0, "score"], bins=25, color=C_OK, alpha=.85, label="APPROVE")
ax.hist(df.loc[df.decision == 1, "score"], bins=25, color=C_BAD, alpha=.9, label="DECLINE")
ax.axvline(THRESHOLD, color="black", ls="--", label=f"Threshold {THRESHOLD}")
ax.set_xlabel("Score"); ax.set_ylabel("Queries")
ax.set_title("Score Distribution in the Query Log"); ax.legend()
finish(fig, "g01_score_distribution.png")

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, f in zip(axes.ravel(), numeric_features):
    ax.scatter(df.loc[df.decision == 0, f], df.loc[df.decision == 0, "score"],
               c=C_OK, s=28, alpha=.8, label="APPROVE")
    ax.scatter(df.loc[df.decision == 1, f], df.loc[df.decision == 1, "score"],
               c=C_BAD, s=40, label="DECLINE", edgecolor="black", lw=.4)
    ax.axhline(THRESHOLD, color="black", ls="--", lw=1)
    ax.set_xlabel(f); ax.set_ylabel("score"); ax.set_title(f"{f} vs score")
axes[0, 0].legend()
fig.suptitle("How each input relates to the score (raw data)", fontweight="bold", y=1.01)
finish(fig, "g02_features_vs_score.png")

corr = df[numeric_features + ["score"]].corr()
fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=35, ha="right")
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns)
ax.grid(False)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=9)
fig.colorbar(im, fraction=.046); ax.set_title("Correlation Heatmap")
finish(fig, "g03_correlation.png")


# ================================================================
# 3. SPLIT + PREPROCESS
# ================================================================
banner("3. SPLIT + PREPROCESSING")
strat = y_dec if np.bincount(y_dec).min() >= 2 else None
idx = np.arange(len(df))
tr_idx, te_idx = train_test_split(idx, test_size=0.20, stratify=strat, random_state=SEED)

X_train, X_test = X.iloc[tr_idx], X.iloc[te_idx]
ys_train, ys_test = y_score[tr_idx], y_score[te_idx]
yd_train, yd_test = y_dec[tr_idx], y_dec[te_idx]
is_test = np.zeros(len(df), dtype=bool); is_test[te_idx] = True

preprocessor = ColumnTransformer([
    ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median")),
                          ("scaler", StandardScaler())]), numeric_features)
])
Xtr = preprocessor.fit_transform(X_train)       # fit on TRAIN only
Xte = preprocessor.transform(X_test)

print(pd.DataFrame({
    "Set": ["Train", "Test"],
    "Rows": [len(tr_idx), len(te_idx)],
    "DECLINE rows": [int(yd_train.sum()), int(yd_test.sum())],
}).to_string(index=False))
print(f"\nModel input dimensions: {Xtr.shape[1]}  ({', '.join(numeric_features)})")


# ================================================================
# 4. MODEL
# ================================================================
banner("4. MODEL")
tf.keras.backend.clear_session()
nn = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(Xtr.shape[1],)),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.10),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.05),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid"),      # score in 0..1
])
nn.compile(optimizer=tf.keras.optimizers.Adam(0.003), loss="mse", metrics=["mae"])
print("Architecture : 4 -> 64 -> 32 -> 16 -> 1 (sigmoid)")
print(f"Parameters   : {nn.count_params():,}")


# ================================================================
# 5. TRAIN (one run, short log)
# ================================================================
banner("5. TRAINING")

class Brief(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        if epoch == 0 or (epoch + 1) % 20 == 0:
            print(f"  epoch {epoch+1:3d} | loss {logs['loss']:.5f} | "
                  f"val_loss {logs['val_loss']:.5f} | val_mae {logs['val_mae']:.4f}")

history = nn.fit(
    Xtr, ys_train, validation_split=0.15, epochs=300, batch_size=16, verbose=0,
    callbacks=[
        Brief(),
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=30,
                                         restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                             patience=10, min_lr=1e-5),
    ],
)
h = history.history
best_ep = int(np.argmin(h["val_loss"])) + 1
print(f"\n  stopped at epoch {len(h['loss'])} | best epoch {best_ep} "
      f"| best val_loss {min(h['val_loss']):.5f}")


# ================================================================
# 6. PREDICTIONS
# ================================================================
def predict_scores(frame):
    xp = preprocessor.transform(frame[numeric_features])
    return np.clip(nn.predict(xp, verbose=0).ravel(), 0.0, 1.0)

pred_test = predict_scores(X_test)
pred_all = predict_scores(X)


# ================================================================
# 7. METRICS TABLE
# ================================================================
banner("6. RESULTS")

def evaluate(ys, ps, yd):
    yp = (ps < THRESHOLD).astype(int)
    both = len(np.unique(yd)) == 2
    return {
        "MAE (score)":        mean_absolute_error(ys, ps),
        "RMSE (score)":       float(np.sqrt(mean_squared_error(ys, ps))),
        "R2 (score)":         r2_score(ys, ps) if len(ys) > 1 else float("nan"),
        "Accuracy":           accuracy_score(yd, yp),
        "Precision DECLINE":  precision_score(yd, yp, zero_division=0),
        "Recall DECLINE":     recall_score(yd, yp, zero_division=0),
        "F1 DECLINE":         f1_score(yd, yp, zero_division=0),
        "ROC-AUC":            roc_auc_score(yd, 1 - ps) if both else float("nan"),
        "Average Precision":  average_precision_score(yd, 1 - ps) if both else float("nan"),
    }

m_test = evaluate(ys_test, pred_test, yd_test)
m_all = evaluate(y_score, pred_all, y_dec)

print(f"{'Metric':<22}{'Test (' + str(len(te_idx)) + ' unseen)':>18}{'All data (in-sample)':>24}")
print("-" * 64)
for k in m_test:
    print(f"{k:<22}{fmt(m_test[k]):>18}{fmt(m_all[k]):>24}")
print("-" * 64)
print(f"Decision rule: score < {THRESHOLD} -> DECLINE")
if int(yd_test.sum()) < 5:
    print(f"\n  ! Only {int(yd_test.sum())} DECLINE row(s) in the test set - decision metrics on "
          "the test column\n    are not reliable. Add more DECLINE queries to the log.")

yp_test = (pred_test < THRESHOLD).astype(int)
cm = confusion_matrix(yd_test, yp_test, labels=[0, 1])
print("\nConfusion matrix (test)      pred APPROVE   pred DECLINE")
print(f"  actual APPROVE             {cm[0,0]:>12}   {cm[0,1]:>12}")
print(f"  actual DECLINE             {cm[1,0]:>12}   {cm[1,1]:>12}")


# ================================================================
# 8. SENSITIVITY + IMPORTANCE
# ================================================================
med = X.median()
sens_rows, pd_curves = [], {}
for f in numeric_features:
    lo, hi = SCHEMA[f]
    grid_vals = np.linspace(lo, hi, 100)
    g = pd.DataFrame({c: np.full(100, med[c]) for c in numeric_features})
    g[f] = grid_vals
    sc = predict_scores(g)
    pd_curves[f] = (grid_vals, sc)
    sens_rows.append({"Feature": f, "Score @ min": sc[0], "Score @ max": sc[-1],
                      "Change": sc[-1] - sc[0]})
sens = pd.DataFrame(sens_rows).set_index("Feature")

rng = np.random.default_rng(SEED)
Xall_p = preprocessor.transform(X)
base_mae = mean_absolute_error(y_score, np.clip(nn.predict(Xall_p, verbose=0).ravel(), 0, 1))
imp = {}
for j, f in enumerate(numeric_features):
    deltas = []
    for _ in range(20):
        Xp = Xall_p.copy(); rng.shuffle(Xp[:, j])
        deltas.append(mean_absolute_error(
            y_score, np.clip(nn.predict(Xp, verbose=0).ravel(), 0, 1)) - base_mae)
    imp[f] = (np.mean(deltas), np.std(deltas))
imp = pd.DataFrame(imp, index=["importance", "std"]).T.sort_values("importance", ascending=False)

print("\nSENSITIVITY  (one input swept over its full range, others at the median)")
print(sens.round(4).to_string())
print("\nFEATURE IMPORTANCE  (rise in MAE when the feature is shuffled)")
print(imp.round(4).to_string())

top = imp.index[0]
print("\nKEY TAKEAWAYS")
print(f"  - Score error on unseen data: MAE {fmt(m_test['MAE (score)'])} (R2 {fmt(m_test['R2 (score)'])}).")
print(f"  - Most influential input: {top}.")
for f in numeric_features:
    d = sens.loc[f, "Change"]
    print(f"  - Raising {f:<17}: score {'rises' if d > 0 else 'falls'} by {abs(d):.3f} over its range.")


# ================================================================
# 9. MODEL GRAPHS
# ================================================================
banner("7. MODEL GRAPHS")

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.plot(h["loss"], lw=2.4, color=C_MAIN, label="Training Loss")
ax.plot(h["val_loss"], lw=2.4, color=C_ALT, label="Validation Loss")
ax.axvline(best_ep - 1, color="gray", ls=":", label=f"Best epoch ({best_ep})")
ax.set_yscale("log"); ax.set_xlabel("Epoch"); ax.set_ylabel("MSE (log scale)")
ax.set_title("Training & Validation Loss"); ax.legend()
finish(fig, "g04_loss.png")

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.plot(h["mae"], lw=2.4, color=C_MAIN, label="Training MAE")
ax.plot(h["val_mae"], lw=2.4, color=C_ALT, label="Validation MAE")
ax.set_xlabel("Epoch"); ax.set_ylabel("MAE (score units)")
ax.set_title("Training & Validation MAE"); ax.legend()
finish(fig, "g05_mae.png")

fig, ax = plt.subplots(figsize=(7, 6.5))
ax.scatter(y_score[~is_test], pred_all[~is_test], s=30, c=C_MAIN, alpha=.6, label="Train")
ax.scatter(y_score[is_test], pred_all[is_test], s=55, c=C_BAD, edgecolor="black",
           lw=.5, label="Test")
ax.plot([0, 1], [0, 1], "--", color="gray")
ax.axhline(THRESHOLD, color="black", lw=.8); ax.axvline(THRESHOLD, color="black", lw=.8)
ax.set_xlabel("Actual score"); ax.set_ylabel("Predicted score")
ax.set_title(f"Actual vs Predicted Score (test MAE {fmt(m_test['MAE (score)'])})"); ax.legend()
finish(fig, "g06_actual_vs_predicted.png")

resid = pred_all - y_score
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].hist(resid, bins=25, color=C_MAIN, alpha=.85)
axes[0].axvline(0, color="red", ls="--")
axes[0].set_xlabel("Predicted - Actual"); axes[0].set_ylabel("Queries")
axes[0].set_title("Residual Distribution")
axes[1].scatter(pred_all, resid, s=28, c=np.where(is_test, C_BAD, C_MAIN), alpha=.8)
axes[1].axhline(0, color="red", ls="--")
axes[1].set_xlabel("Predicted score"); axes[1].set_ylabel("Residual")
axes[1].set_title("Residuals vs Predicted (red = test)")
finish(fig, "g07_residuals.png")

if len(np.unique(yd_test)) == 2:
    ev_y, ev_p, tag = yd_test, 1 - pred_test, "test set"
else:
    ev_y, ev_p, tag = y_dec, 1 - pred_all, "all data, in-sample"

if len(np.unique(ev_y)) == 2:
    fpr, tpr, _ = roc_curve(ev_y, ev_p); auc = roc_auc_score(ev_y, ev_p)
    fig, ax = plt.subplots(figsize=(6.8, 6.2))
    ax.plot(fpr, tpr, lw=3, color=C_MAIN, label=f"TensorFlow (AUC = {auc:.4f})")
    ax.plot([0, 1], [0, 1], "--", color=C_BAD, label="Random")
    ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
    ax.set_title(f"ROC Curve ({tag})"); ax.legend(loc="lower right")
    finish(fig, "g08_roc.png")

    pr, rc, _ = precision_recall_curve(ev_y, ev_p); ap = average_precision_score(ev_y, ev_p)
    fig, ax = plt.subplots(figsize=(6.8, 6.2))
    ax.plot(rc, pr, lw=3, color=C_MAIN, label=f"TensorFlow (AP = {ap:.4f})")
    ax.axhline(ev_y.mean(), ls="--", color=C_BAD, label=f"Baseline = {ev_y.mean():.3f}")
    ax.set_ylim(0, 1.05); ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
    ax.set_title(f"Precision-Recall Curve ({tag})"); ax.legend(loc="lower left")
    finish(fig, "g09_pr.png")

fig, ax = plt.subplots(figsize=(6.5, 5.8))
ax.imshow(cm, cmap="Blues"); ax.grid(False)
ax.set_xticks([0, 1]); ax.set_xticklabels(["APPROVE", "DECLINE"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["APPROVE", "DECLINE"])
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title("Confusion Matrix (test)")
for i in range(2):
    rs = cm[i].sum()
    for j in range(2):
        pct = f"{100*cm[i,j]/rs:.0f}%" if rs else "-"
        ax.text(j, i, f"{cm[i,j]}\n{pct}", ha="center", va="center", fontsize=16,
                fontweight="bold", color="white" if cm[i, j] > cm.max() / 2 else "black")
finish(fig, "g10_confusion.png")

names = ["Accuracy", "Precision DECLINE", "Recall DECLINE", "F1 DECLINE"]
x = np.arange(len(names)); w = .38
fig, ax = plt.subplots(figsize=(9.5, 5.2))
b1 = ax.bar(x - w/2, [m_test[n] for n in names], w, color=C_MAIN, label="Test")
b2 = ax.bar(x + w/2, [m_all[n] for n in names], w, color=C_ALT, label="All data (in-sample)")
for b in list(b1) + list(b2):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + .015, f"{b.get_height():.2f}",
            ha="center", fontsize=9, fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(names); ax.set_ylim(0, 1.1)
ax.set_title("Classification Performance"); ax.legend()
finish(fig, "g11_metrics.png")

fig, ax = plt.subplots(figsize=(8, 4.6))
imp_r = imp.iloc[::-1]
ax.barh(imp_r.index, imp_r["importance"], xerr=imp_r["std"], color=C_MAIN, alpha=.9)
ax.set_xlabel("Increase in MAE when shuffled")
ax.set_title("Feature Importance (permutation)")
finish(fig, "g12_importance.png")

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, f in zip(axes.ravel(), numeric_features):
    gx, sc = pd_curves[f]
    ax.plot(gx, sc, lw=2.8, color=C_MAIN)
    ax.axhline(THRESHOLD, color=C_BAD, ls="--", label="Threshold")
    ax.fill_between(gx, 0, sc, where=sc < THRESHOLD, color=C_BAD, alpha=.2)
    ax.set_ylim(0, 1.02); ax.set_xlabel(f); ax.set_ylabel("Predicted score")
    ax.set_title(f"Effect of {f}")
axes[0, 0].legend()
fig.suptitle("Sensitivity: one input swept, others at median", fontweight="bold", y=1.01)
finish(fig, "g13_sensitivity.png")

def decision_map(fx, fy, name):
    gx = np.linspace(*SCHEMA[fx], 60); gy = np.linspace(*SCHEMA[fy], 60)
    GX, GY = np.meshgrid(gx, gy)
    g = pd.DataFrame({c: np.full(GX.size, med[c]) for c in numeric_features})
    g[fx] = GX.ravel(); g[fy] = GY.ravel()
    Z = predict_scores(g).reshape(GX.shape)
    fig, ax = plt.subplots(figsize=(7.8, 6.2))
    cf = ax.contourf(GX, GY, Z, levels=np.linspace(0, 1, 21), cmap="RdYlGn")
    ax.contour(GX, GY, Z, levels=[THRESHOLD], colors="black", linewidths=2.2)
    ax.scatter(df.loc[df.decision == 0, fx], df.loc[df.decision == 0, fy],
               s=14, c="white", edgecolor="gray", lw=.4, label="APPROVE (data)")
    ax.scatter(df.loc[df.decision == 1, fx], df.loc[df.decision == 1, fy],
               s=40, c="black", marker="X", label="DECLINE (data)")
    ax.set_xlabel(fx); ax.set_ylabel(fy)
    ax.set_title(f"Predicted Score Map: {fx} vs {fy}\n(black line = decision boundary)")
    ax.legend(loc="upper right", fontsize=8)
    fig.colorbar(cf, label="Predicted score")
    finish(fig, name)

decision_map("reach", "report_ratio", "g14_map_reach_report.png")
decision_map("account_age_days", "months_active", "g15_map_age_months.png")


# ================================================================
# 10. SAVE THE TRAINED MODEL  (3 files = the model)
# ================================================================
banner("8. SAVED FILES")

res = X.copy()
res["Split"] = np.where(is_test, "test", "train")
res["Actual_Score"] = y_score
res["Actual_Decision"] = np.where(y_dec == 1, "DECLINE", "APPROVE")
res["TF_Score"] = pred_all.round(4)
res["TF_Decision"] = np.where(pred_all < THRESHOLD, "DECLINE", "APPROVE")
res["Abs_Error"] = np.abs(pred_all - y_score).round(4)
res.to_csv(os.path.join(OUT_DIR, "TENSORFLOW_FINAL_RESULTS.csv"), index=False)

pd.DataFrame({"Test": m_test, "All_data_in_sample": m_all}).to_csv(
    os.path.join(OUT_DIR, "metrics_summary.csv"))

MODEL_PATH = os.path.join(OUT_DIR, "gk04_tf_model.keras")
PREP_PATH = os.path.join(OUT_DIR, "gk04_preprocessor.joblib")
CONF_PATH = os.path.join(OUT_DIR, "gk04_config.json")

nn.save(MODEL_PATH)                                   # the network
joblib.dump(preprocessor, PREP_PATH)                  # the scaler
with open(CONF_PATH, "w") as fh:                      # ranges + threshold
    json.dump({"features": numeric_features,
               "schema": SCHEMA,
               "threshold": THRESHOLD}, fh, indent=2)

print("TRAINED MODEL (needed together):")
print(f"  {MODEL_PATH}")
print(f"  {PREP_PATH}")
print(f"  {CONF_PATH}")
print("\nREPORT FILES:")
print(f"  {OUT_DIR}/TENSORFLOW_FINAL_RESULTS.csv   (all rows, with Split column)")
print(f"  {OUT_DIR}/metrics_summary.csv")
print(f"  {OUT_DIR}/g01 ... g15 *.png               (graphs)")


# ================================================================
# 11. QUERY THE REPLICA (portal-style: all fields required)
# ================================================================
def query(reach, account_age_days, months_active, report_ratio):
    row = dict(reach=reach, account_age_days=account_age_days,
               months_active=months_active, report_ratio=report_ratio)
    for k, v in row.items():
        lo, hi = SCHEMA[k]
        if v is None or not (lo <= float(v) <= hi):
            raise ValueError(f"{k}={v} is outside the allowed range {lo}-{hi}")
    s = float(predict_scores(pd.DataFrame([row]))[0])
    return {"score": round(s, 4), "decision": "APPROVE" if s >= THRESHOLD else "DECLINE"}

banner("9. EXAMPLE QUERIES")
examples = [
    dict(reach=0,   account_age_days=75, months_active=0,  report_ratio=0.0),
    dict(reach=50,  account_age_days=50, months_active=20, report_ratio=0.5),
    dict(reach=100, account_age_days=18, months_active=40, report_ratio=1.0),
]
print(pd.DataFrame([{**e, **query(**e)} for e in examples]).to_string(index=False))

In [ ]:
from google.colab import files
files.upload()   # pick gk04_queries.csv


In [ ]:
# ================================================================
# GK-04 CONTENT MODERATION  ->  TENSORFLOW REPLICA  (clean report)
# Input : reach, account_age_days, months_active, report_ratio
# Output: score (0-1) and decision (APPROVE / DECLINE)
# Needs : /content/gk04_queries.csv with columns
#         reach, account_age_days, months_active, report_ratio, score, decision
# Saves : gk04_tf_model.keras + gk04_preprocessor.joblib + gk04_config.json
#         (these three files together = the trained model)
# ================================================================
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import json, random, warnings
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, average_precision_score, precision_recall_curve,
    confusion_matrix, mean_absolute_error, mean_squared_error, r2_score,
)

warnings.filterwarnings("ignore")

# ---------------- SETTINGS ----------------
SEED = 42
FILE = "/content/gk04_queries.csv"
OUT_DIR = "/content"
if not os.path.isdir(OUT_DIR):            # running locally
    OUT_DIR, FILE = ".", "gk04_queries.csv"

SCHEMA = {
    "reach":            (0.0, 100.0),
    "account_age_days": (18.0, 75.0),
    "months_active":    (0.0, 40.0),
    "report_ratio":     (0.0, 1.0),
}
numeric_features = list(SCHEMA)
THRESHOLD = 0.527        # portal: 0.5099 = DECLINE, 0.5436 = APPROVE

C_OK, C_BAD, C_MAIN, C_ALT = "#2a9d8f", "#e76f51", "#264653", "#e9c46a"

np.random.seed(SEED); random.seed(SEED); tf.random.set_seed(SEED)
plt.rcParams.update({
    "figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titleweight": "bold", "axes.titlesize": 13,
})

def banner(t):
    print("\n" + "=" * 70 + f"\n{t}\n" + "=" * 70)

def fmt(v):
    return "n/a" if v is None or (isinstance(v, float) and np.isnan(v)) else f"{v:.4f}"

def finish(fig, name):
    fig.tight_layout()
    path = os.path.join(OUT_DIR, name)
    fig.savefig(path, dpi=140)
    plt.show()
    print(f"   saved -> {path}")


# ================================================================
# 1. LOAD + CLEAN
# ================================================================
banner("1. DATA")
raw = pd.read_csv(FILE)
need = numeric_features + ["score", "decision"]
miss = [c for c in need if c not in raw.columns]
if miss:
    raise ValueError(f"CSV is missing required columns: {miss}")

df = raw[need].drop_duplicates().reset_index(drop=True)

if df["decision"].dtype == "object":
    df["decision"] = (df["decision"].astype(str).str.strip().str.upper()
                      .map({"APPROVE": 0, "DECLINE": 1}))
df["decision"] = pd.to_numeric(df["decision"], errors="coerce")
df["score"] = pd.to_numeric(df["score"], errors="coerce")
if df[["decision", "score"]].isna().any().any():
    raise ValueError("'score' / 'decision' contain invalid or missing values.")
df["decision"] = df["decision"].astype(int)

n_conf = int((df.groupby(numeric_features)["score"].nunique() > 1).sum())
info = pd.Series({
    "Rows in file": len(raw),
    "Unique rows used": len(df),
    "APPROVE rows": int((df.decision == 0).sum()),
    "DECLINE rows": int((df.decision == 1).sum()),
    "Ignored columns": ", ".join(c for c in raw.columns if c not in need) or "none",
    "Conflicting input combos": n_conf,
})
print(info.to_string())
if n_conf:
    print("\n  ! Same inputs gave different scores (hidden fields were varying)."
          "\n    Expect a noise floor on the error.")

print("\nFeature summary:")
print(df[numeric_features + ["score"]].describe().T[["min", "mean", "max"]].round(3).to_string())

X = df[numeric_features]
y_score = df["score"].values
y_dec = df["decision"].values


# ================================================================
# 2. EXPLORATORY GRAPHS
# ================================================================
banner("2. DATA GRAPHS")

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(df.loc[df.decision == 0, "score"], bins=25, color=C_OK, alpha=.85, label="APPROVE")
ax.hist(df.loc[df.decision == 1, "score"], bins=25, color=C_BAD, alpha=.9, label="DECLINE")
ax.axvline(THRESHOLD, color="black", ls="--", label=f"Threshold {THRESHOLD}")
ax.set_xlabel("Score"); ax.set_ylabel("Queries")
ax.set_title("Score Distribution in the Query Log"); ax.legend()
finish(fig, "g01_score_distribution.png")

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, f in zip(axes.ravel(), numeric_features):
    ax.scatter(df.loc[df.decision == 0, f], df.loc[df.decision == 0, "score"],
               c=C_OK, s=28, alpha=.8, label="APPROVE")
    ax.scatter(df.loc[df.decision == 1, f], df.loc[df.decision == 1, "score"],
               c=C_BAD, s=40, label="DECLINE", edgecolor="black", lw=.4)
    ax.axhline(THRESHOLD, color="black", ls="--", lw=1)
    ax.set_xlabel(f); ax.set_ylabel("score"); ax.set_title(f"{f} vs score")
axes[0, 0].legend()
fig.suptitle("How each input relates to the score (raw data)", fontweight="bold", y=1.01)
finish(fig, "g02_features_vs_score.png")

corr = df[numeric_features + ["score"]].corr()
fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=35, ha="right")
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns)
ax.grid(False)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=9)
fig.colorbar(im, fraction=.046); ax.set_title("Correlation Heatmap")
finish(fig, "g03_correlation.png")


# ================================================================
# 3. SPLIT + PREPROCESS
# ================================================================
banner("3. SPLIT + PREPROCESSING")
strat = y_dec if np.bincount(y_dec).min() >= 2 else None
idx = np.arange(len(df))
tr_idx, te_idx = train_test_split(idx, test_size=0.20, stratify=strat, random_state=SEED)

X_train, X_test = X.iloc[tr_idx], X.iloc[te_idx]
ys_train, ys_test = y_score[tr_idx], y_score[te_idx]
yd_train, yd_test = y_dec[tr_idx], y_dec[te_idx]
is_test = np.zeros(len(df), dtype=bool); is_test[te_idx] = True

preprocessor = ColumnTransformer([
    ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median")),
                          ("scaler", StandardScaler())]), numeric_features)
])
Xtr = preprocessor.fit_transform(X_train)       # fit on TRAIN only
Xte = preprocessor.transform(X_test)

print(pd.DataFrame({
    "Set": ["Train", "Test"],
    "Rows": [len(tr_idx), len(te_idx)],
    "DECLINE rows": [int(yd_train.sum()), int(yd_test.sum())],
}).to_string(index=False))
print(f"\nModel input dimensions: {Xtr.shape[1]}  ({', '.join(numeric_features)})")


# ================================================================
# 4. MODEL
# ================================================================
banner("4. MODEL")
tf.keras.backend.clear_session()
nn = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(Xtr.shape[1],)),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.10),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.05),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid"),      # score in 0..1
])
nn.compile(optimizer=tf.keras.optimizers.Adam(0.003), loss="mse", metrics=["mae"])
print("Architecture : 4 -> 64 -> 32 -> 16 -> 1 (sigmoid)")
print(f"Parameters   : {nn.count_params():,}")


# ================================================================
# 5. TRAIN (one run, short log)
# ================================================================
banner("5. TRAINING")

class Brief(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        if epoch == 0 or (epoch + 1) % 20 == 0:
            print(f"  epoch {epoch+1:3d} | loss {logs['loss']:.5f} | "
                  f"val_loss {logs['val_loss']:.5f} | val_mae {logs['val_mae']:.4f}")

history = nn.fit(
    Xtr, ys_train, validation_split=0.15, epochs=300, batch_size=16, verbose=0,
    callbacks=[
        Brief(),
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=30,
                                         restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                             patience=10, min_lr=1e-5),
    ],
)
h = history.history
best_ep = int(np.argmin(h["val_loss"])) + 1
print(f"\n  stopped at epoch {len(h['loss'])} | best epoch {best_ep} "
      f"| best val_loss {min(h['val_loss']):.5f}")


# ================================================================
# 6. PREDICTIONS
# ================================================================
def predict_scores(frame):
    xp = preprocessor.transform(frame[numeric_features])
    return np.clip(nn.predict(xp, verbose=0).ravel(), 0.0, 1.0)

pred_test = predict_scores(X_test)
pred_all = predict_scores(X)


# ================================================================
# 7. METRICS TABLE
# ================================================================
banner("6. RESULTS")

def evaluate(ys, ps, yd):
    yp = (ps < THRESHOLD).astype(int)
    both = len(np.unique(yd)) == 2
    return {
        "MAE (score)":        mean_absolute_error(ys, ps),
        "RMSE (score)":       float(np.sqrt(mean_squared_error(ys, ps))),
        "R2 (score)":         r2_score(ys, ps) if len(ys) > 1 else float("nan"),
        "Accuracy":           accuracy_score(yd, yp),
        "Precision DECLINE":  precision_score(yd, yp, zero_division=0),
        "Recall DECLINE":     recall_score(yd, yp, zero_division=0),
        "F1 DECLINE":         f1_score(yd, yp, zero_division=0),
        "ROC-AUC":            roc_auc_score(yd, 1 - ps) if both else float("nan"),
        "Average Precision":  average_precision_score(yd, 1 - ps) if both else float("nan"),
    }

m_test = evaluate(ys_test, pred_test, yd_test)
m_all = evaluate(y_score, pred_all, y_dec)

print(f"{'Metric':<22}{'Test (' + str(len(te_idx)) + ' unseen)':>18}{'All data (in-sample)':>24}")
print("-" * 64)
for k in m_test:
    print(f"{k:<22}{fmt(m_test[k]):>18}{fmt(m_all[k]):>24}")
print("-" * 64)
print(f"Decision rule: score < {THRESHOLD} -> DECLINE")
if int(yd_test.sum()) < 5:
    print(f"\n  ! Only {int(yd_test.sum())} DECLINE row(s) in the test set - decision metrics on "
          "the test column\n    are not reliable. Add more DECLINE queries to the log.")

yp_test = (pred_test < THRESHOLD).astype(int)
cm = confusion_matrix(yd_test, yp_test, labels=[0, 1])
print("\nConfusion matrix (test)      pred APPROVE   pred DECLINE")
print(f"  actual APPROVE             {cm[0,0]:>12}   {cm[0,1]:>12}")
print(f"  actual DECLINE             {cm[1,0]:>12}   {cm[1,1]:>12}")


# ================================================================
# 8. SENSITIVITY + IMPORTANCE
# ================================================================
med = X.median()
sens_rows, pd_curves = [], {}
for f in numeric_features:
    lo, hi = SCHEMA[f]
    grid_vals = np.linspace(lo, hi, 100)
    g = pd.DataFrame({c: np.full(100, med[c]) for c in numeric_features})
    g[f] = grid_vals
    sc = predict_scores(g)
    pd_curves[f] = (grid_vals, sc)
    sens_rows.append({"Feature": f, "Score @ min": sc[0], "Score @ max": sc[-1],
                      "Change": sc[-1] - sc[0]})
sens = pd.DataFrame(sens_rows).set_index("Feature")

rng = np.random.default_rng(SEED)
Xall_p = preprocessor.transform(X)
base_mae = mean_absolute_error(y_score, np.clip(nn.predict(Xall_p, verbose=0).ravel(), 0, 1))
imp = {}
for j, f in enumerate(numeric_features):
    deltas = []
    for _ in range(20):
        Xp = Xall_p.copy(); rng.shuffle(Xp[:, j])
        deltas.append(mean_absolute_error(
            y_score, np.clip(nn.predict(Xp, verbose=0).ravel(), 0, 1)) - base_mae)
    imp[f] = (np.mean(deltas), np.std(deltas))
imp = pd.DataFrame(imp, index=["importance", "std"]).T.sort_values("importance", ascending=False)

print("\nSENSITIVITY  (one input swept over its full range, others at the median)")
print(sens.round(4).to_string())
print("\nFEATURE IMPORTANCE  (rise in MAE when the feature is shuffled)")
print(imp.round(4).to_string())

top = imp.index[0]
print("\nKEY TAKEAWAYS")
print(f"  - Score error on unseen data: MAE {fmt(m_test['MAE (score)'])} (R2 {fmt(m_test['R2 (score)'])}).")
print(f"  - Most influential input: {top}.")
for f in numeric_features:
    d = sens.loc[f, "Change"]
    print(f"  - Raising {f:<17}: score {'rises' if d > 0 else 'falls'} by {abs(d):.3f} over its range.")


# ================================================================
# 9. MODEL GRAPHS
# ================================================================
banner("7. MODEL GRAPHS")

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.plot(h["loss"], lw=2.4, color=C_MAIN, label="Training Loss")
ax.plot(h["val_loss"], lw=2.4, color=C_ALT, label="Validation Loss")
ax.axvline(best_ep - 1, color="gray", ls=":", label=f"Best epoch ({best_ep})")
ax.set_yscale("log"); ax.set_xlabel("Epoch"); ax.set_ylabel("MSE (log scale)")
ax.set_title("Training & Validation Loss"); ax.legend()
finish(fig, "g04_loss.png")

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.plot(h["mae"], lw=2.4, color=C_MAIN, label="Training MAE")
ax.plot(h["val_mae"], lw=2.4, color=C_ALT, label="Validation MAE")
ax.set_xlabel("Epoch"); ax.set_ylabel("MAE (score units)")
ax.set_title("Training & Validation MAE"); ax.legend()
finish(fig, "g05_mae.png")

fig, ax = plt.subplots(figsize=(7, 6.5))
ax.scatter(y_score[~is_test], pred_all[~is_test], s=30, c=C_MAIN, alpha=.6, label="Train")
ax.scatter(y_score[is_test], pred_all[is_test], s=55, c=C_BAD, edgecolor="black",
           lw=.5, label="Test")
ax.plot([0, 1], [0, 1], "--", color="gray")
ax.axhline(THRESHOLD, color="black", lw=.8); ax.axvline(THRESHOLD, color="black", lw=.8)
ax.set_xlabel("Actual score"); ax.set_ylabel("Predicted score")
ax.set_title(f"Actual vs Predicted Score (test MAE {fmt(m_test['MAE (score)'])})"); ax.legend()
finish(fig, "g06_actual_vs_predicted.png")

resid = pred_all - y_score
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].hist(resid, bins=25, color=C_MAIN, alpha=.85)
axes[0].axvline(0, color="red", ls="--")
axes[0].set_xlabel("Predicted - Actual"); axes[0].set_ylabel("Queries")
axes[0].set_title("Residual Distribution")
axes[1].scatter(pred_all, resid, s=28, c=np.where(is_test, C_BAD, C_MAIN), alpha=.8)
axes[1].axhline(0, color="red", ls="--")
axes[1].set_xlabel("Predicted score"); axes[1].set_ylabel("Residual")
axes[1].set_title("Residuals vs Predicted (red = test)")
finish(fig, "g07_residuals.png")

if len(np.unique(yd_test)) == 2:
    ev_y, ev_p, tag = yd_test, 1 - pred_test, "test set"
else:
    ev_y, ev_p, tag = y_dec, 1 - pred_all, "all data, in-sample"

if len(np.unique(ev_y)) == 2:
    fpr, tpr, _ = roc_curve(ev_y, ev_p); auc = roc_auc_score(ev_y, ev_p)
    fig, ax = plt.subplots(figsize=(6.8, 6.2))
    ax.plot(fpr, tpr, lw=3, color=C_MAIN, label=f"TensorFlow (AUC = {auc:.4f})")
    ax.plot([0, 1], [0, 1], "--", color=C_BAD, label="Random")
    ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
    ax.set_title(f"ROC Curve ({tag})"); ax.legend(loc="lower right")
    finish(fig, "g08_roc.png")

    pr, rc, _ = precision_recall_curve(ev_y, ev_p); ap = average_precision_score(ev_y, ev_p)
    fig, ax = plt.subplots(figsize=(6.8, 6.2))
    ax.plot(rc, pr, lw=3, color=C_MAIN, label=f"TensorFlow (AP = {ap:.4f})")
    ax.axhline(ev_y.mean(), ls="--", color=C_BAD, label=f"Baseline = {ev_y.mean():.3f}")
    ax.set_ylim(0, 1.05); ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
    ax.set_title(f"Precision-Recall Curve ({tag})"); ax.legend(loc="lower left")
    finish(fig, "g09_pr.png")

fig, ax = plt.subplots(figsize=(6.5, 5.8))
ax.imshow(cm, cmap="Blues"); ax.grid(False)
ax.set_xticks([0, 1]); ax.set_xticklabels(["APPROVE", "DECLINE"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["APPROVE", "DECLINE"])
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title("Confusion Matrix (test)")
for i in range(2):
    rs = cm[i].sum()
    for j in range(2):
        pct = f"{100*cm[i,j]/rs:.0f}%" if rs else "-"
        ax.text(j, i, f"{cm[i,j]}\n{pct}", ha="center", va="center", fontsize=16,
                fontweight="bold", color="white" if cm[i, j] > cm.max() / 2 else "black")
finish(fig, "g10_confusion.png")

names = ["Accuracy", "Precision DECLINE", "Recall DECLINE", "F1 DECLINE"]
x = np.arange(len(names)); w = .38
fig, ax = plt.subplots(figsize=(9.5, 5.2))
b1 = ax.bar(x - w/2, [m_test[n] for n in names], w, color=C_MAIN, label="Test")
b2 = ax.bar(x + w/2, [m_all[n] for n in names], w, color=C_ALT, label="All data (in-sample)")
for b in list(b1) + list(b2):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + .015, f"{b.get_height():.2f}",
            ha="center", fontsize=9, fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(names); ax.set_ylim(0, 1.1)
ax.set_title("Classification Performance"); ax.legend()
finish(fig, "g11_metrics.png")

fig, ax = plt.subplots(figsize=(8, 4.6))
imp_r = imp.iloc[::-1]
ax.barh(imp_r.index, imp_r["importance"], xerr=imp_r["std"], color=C_MAIN, alpha=.9)
ax.set_xlabel("Increase in MAE when shuffled")
ax.set_title("Feature Importance (permutation)")
finish(fig, "g12_importance.png")

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, f in zip(axes.ravel(), numeric_features):
    gx, sc = pd_curves[f]
    ax.plot(gx, sc, lw=2.8, color=C_MAIN)
    ax.axhline(THRESHOLD, color=C_BAD, ls="--", label="Threshold")
    ax.fill_between(gx, 0, sc, where=sc < THRESHOLD, color=C_BAD, alpha=.2)
    ax.set_ylim(0, 1.02); ax.set_xlabel(f); ax.set_ylabel("Predicted score")
    ax.set_title(f"Effect of {f}")
axes[0, 0].legend()
fig.suptitle("Sensitivity: one input swept, others at median", fontweight="bold", y=1.01)
finish(fig, "g13_sensitivity.png")

def decision_map(fx, fy, name):
    gx = np.linspace(*SCHEMA[fx], 60); gy = np.linspace(*SCHEMA[fy], 60)
    GX, GY = np.meshgrid(gx, gy)
    g = pd.DataFrame({c: np.full(GX.size, med[c]) for c in numeric_features})
    g[fx] = GX.ravel(); g[fy] = GY.ravel()
    Z = predict_scores(g).reshape(GX.shape)
    fig, ax = plt.subplots(figsize=(7.8, 6.2))
    cf = ax.contourf(GX, GY, Z, levels=np.linspace(0, 1, 21), cmap="RdYlGn")
    ax.contour(GX, GY, Z, levels=[THRESHOLD], colors="black", linewidths=2.2)
    ax.scatter(df.loc[df.decision == 0, fx], df.loc[df.decision == 0, fy],
               s=14, c="white", edgecolor="gray", lw=.4, label="APPROVE (data)")
    ax.scatter(df.loc[df.decision == 1, fx], df.loc[df.decision == 1, fy],
               s=40, c="black", marker="X", label="DECLINE (data)")
    ax.set_xlabel(fx); ax.set_ylabel(fy)
    ax.set_title(f"Predicted Score Map: {fx} vs {fy}\n(black line = decision boundary)")
    ax.legend(loc="upper right", fontsize=8)
    fig.colorbar(cf, label="Predicted score")
    finish(fig, name)

decision_map("reach", "report_ratio", "g14_map_reach_report.png")
decision_map("account_age_days", "months_active", "g15_map_age_months.png")


# ================================================================
# 10. SAVE THE TRAINED MODEL  (3 files = the model)
# ================================================================
banner("8. SAVED FILES")

res = X.copy()
res["Split"] = np.where(is_test, "test", "train")
res["Actual_Score"] = y_score
res["Actual_Decision"] = np.where(y_dec == 1, "DECLINE", "APPROVE")
res["TF_Score"] = pred_all.round(4)
res["TF_Decision"] = np.where(pred_all < THRESHOLD, "DECLINE", "APPROVE")
res["Abs_Error"] = np.abs(pred_all - y_score).round(4)
res.to_csv(os.path.join(OUT_DIR, "TENSORFLOW_FINAL_RESULTS.csv"), index=False)

pd.DataFrame({"Test": m_test, "All_data_in_sample": m_all}).to_csv(
    os.path.join(OUT_DIR, "metrics_summary.csv"))

MODEL_PATH = os.path.join(OUT_DIR, "gk04_tf_model.keras")
PREP_PATH = os.path.join(OUT_DIR, "gk04_preprocessor.joblib")
CONF_PATH = os.path.join(OUT_DIR, "gk04_config.json")

nn.save(MODEL_PATH)                                   # the network
joblib.dump(preprocessor, PREP_PATH)                  # the scaler
with open(CONF_PATH, "w") as fh:                      # ranges + threshold
    json.dump({"features": numeric_features,
               "schema": SCHEMA,
               "threshold": THRESHOLD}, fh, indent=2)

print("TRAINED MODEL (needed together):")
print(f"  {MODEL_PATH}")
print(f"  {PREP_PATH}")
print(f"  {CONF_PATH}")
print("\nREPORT FILES:")
print(f"  {OUT_DIR}/TENSORFLOW_FINAL_RESULTS.csv   (all rows, with Split column)")
print(f"  {OUT_DIR}/metrics_summary.csv")
print(f"  {OUT_DIR}/g01 ... g15 *.png               (graphs)")


# ================================================================
# 11. QUERY THE REPLICA (portal-style: all fields required)
# ================================================================
def query(reach, account_age_days, months_active, report_ratio):
    row = dict(reach=reach, account_age_days=account_age_days,
               months_active=months_active, report_ratio=report_ratio)
    for k, v in row.items():
        lo, hi = SCHEMA[k]
        if v is None or not (lo <= float(v) <= hi):
            raise ValueError(f"{k}={v} is outside the allowed range {lo}-{hi}")
    s = float(predict_scores(pd.DataFrame([row]))[0])
    return {"score": round(s, 4), "decision": "APPROVE" if s >= THRESHOLD else "DECLINE"}

banner("9. EXAMPLE QUERIES")
examples = [
    dict(reach=0,   account_age_days=75, months_active=0,  report_ratio=0.0),
    dict(reach=50,  account_age_days=50, months_active=20, report_ratio=0.5),
    dict(reach=100, account_age_days=18, months_active=40, report_ratio=1.0),
]
print(pd.DataFrame([{**e, **query(**e)} for e in examples]).to_string(index=False))